In [67]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [68]:
with open('input.txt', 'r') as f:
    text = f.read()

In [69]:
len(text)

1115394

In [70]:
tokens = sorted(list(set(text)))
vocab_size = len(tokens)

In [71]:
stoi = {s:i for i,s in enumerate(tokens)}
itos = {i:s for i,s in enumerate(tokens)}

In [72]:
encode = lambda s: [stoi[ch] if ch in tokens else 1 for ch in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [73]:
encode("Jayanth")

[22, 39, 63, 39, 52, 58, 46]

In [74]:
decode(encode("\Jayanth"))

' Jayanth'

In [75]:
decode([1])

' '

In [76]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [77]:
len("Jayanth")

7

In [117]:
block_size = 256
n_embd = 64
n_head = 4
n_layer = 6
device = 'cuda' if torch.cuda.is_available() else 'cpu'
learning_rate = 1e-3
batch_size = 64
max_iters = 5000
eval_interval = 500
eval_iters = 200

In [118]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        # head_size: the dimensionality of the query/key/value vectors. 
        # head_Size = n_embd//n_head. concats to form n_embd .
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,head_size)
        q = self.query(x) # (B,T,head_size)
        v = self.value(x) # (B,T,head_size)

        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B,T,head_size) @ (B, head_size, T) -> (B,T,T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)

        out = wei @ v # (B,T,T) @ (B,T,head_size) -> (B,T,head_size)
        return out

In [119]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        #self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        #out = self.dropout(self.proj(out))
        return out

In [120]:
class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

In [121]:
class DecoderBlock(nn.Module):
    """ Decoder block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.ln1(self.sa(x))
        x = x + self.ln2(self.ffwd(x))
        return x

In [122]:
class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[DecoderBlock(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)


    def forward(self, idx, targets=None):
        B,T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T)) # (T,n_embd)
        x = tok_emb + pos_emb # (B,T,n_embd)
        x = self.blocks(x) # (B,T,n_embd)
        x = self.ln_f(x) # (B,T,n_embd)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [123]:
model = GPTLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

0.323649 M parameters


In [124]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [125]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [127]:
import pandas as pd

history_logs = []

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        history_logs.append({
            'step': iter,
            'train_loss': losses['train'],
            'val_loss': losses['val'],
            'exp_type': '6 multi-head blocks'
        })

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


step 0: train loss 4.1682, val loss 4.1702
step 500: train loss 2.2733, val loss 2.2835
step 1000: train loss 1.8643, val loss 1.9810
step 1500: train loss 1.6570, val loss 1.8418
step 2000: train loss 1.5723, val loss 1.7692
step 2500: train loss 1.5372, val loss 1.7470
step 3000: train loss 1.4995, val loss 1.7157
step 3500: train loss 1.4623, val loss 1.6895
step 4000: train loss 1.4450, val loss 1.6745
step 4500: train loss 1.4261, val loss 1.6575
step 4999: train loss 1.4105, val loss 1.6538


In [128]:
df_history = pd.DataFrame(history_logs)

df_history.to_csv('gpt_training_history.csv', mode='a', index=False, header=False)

In [129]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))
#open('more.txt', 'w').write(decode(m.generate(context, max_new_tokens=10000)[0].tolist()))


And, not the crave such I scabse
Daxe, post the gounts friend I content,
To sweet magish pleasance, for not.

First Warwick he do onceile thy to blow that's
When who knows, sweet it sirrange; he would accuse
All begs, like to mercepted to ear recornat?

DUKE Onnerman:
Why, I go! be sit my fallies dull perewp'
To enemnation
That is his off-disxe and the envenge
Were is my wlot: still I might your glase
ques your dall best that matter down.'

BRAKERIO:
Go. You will shall complander for thy best.


